In [1]:
import geopandas as gpd
import geemap
import ee

import os
from glob import glob

In [2]:
# List available cities in mnt/
cities = [d for d in os.listdir('mnt') if os.path.isdir(f'mnt/{d}') and d.startswith('20')]
print("Available cities:")

for i, city in enumerate(cities, 1):
    print(f"{i}. {city}")


Available cities:
1. 2025-10-senegal-tambacounda
2. 2025-10-senegal-diourbel
3. 2025-02-tunisia-tunis
4. 2025-10-indonesia-sofifi
5. 2025-10-senegal-dakar
6. 2025-10-senegal-matam


In [3]:
# Select city
city_dir = cities[4] # Change this to your city

cityname  = city_dir.split('-')[-1] 
data_dir = os.path.join('mnt', city_dir, '02-process-output', 'spatial')
aoi_dir = os.path.join('mnt', city_dir, '01-user-input', 'AOI')
aoi_file = glob(aoi_dir + "/*.shp")[0]

print(f"\n {cityname} - selected: {city_dir}. data_dir: {data_dir}")
print(aoi_file)




 dakar - selected: 2025-10-senegal-dakar. data_dir: mnt/2025-10-senegal-dakar/02-process-output/spatial
mnt/2025-10-senegal-dakar/01-user-input/AOI/senegal_dakar.shp


In [4]:
# Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project = "acr-dev-450519")

In [5]:
# ============================================
# 1. LOAD AOI FROM LOCAL SHAPEFILE
# ============================================
aoi_gdf = gpd.read_file(aoi_file)
aoi = geemap.gdf_to_ee(aoi_gdf)

# Define time range
start_year = 2016
end_year = 2023

In [6]:
criteria = ee.Filter.And([
    ee.Filter.calendarRange(start_year, end_year, 'year'), 
    ee.Filter.bounds(aoi)  
])

In [7]:
def get_yearly_s2(year):
    year = ee.Number(year)
    
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filter(ee.Filter.calendarRange(year, year, 'year'))
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
        .select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'])
        .median()
        .clip(aoi)
        .set('year', year))

def get_yearly_s1(year):
    year = ee.Number(year)
    
    return (ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(aoi)
        .filter(ee.Filter.calendarRange(year, year, 'year'))
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .select(['VV', 'VH'])
        .mean()
        .clip(aoi)
        .set('year', year))

def get_yearly_viirs(year):
    year = ee.Number(year)
    return (ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
        .filterBounds(aoi)
        .filter(ee.Filter.calendarRange(year, year, 'year'))
        .select('avg_rad')
        .min()
        .clip(aoi)
        .set('year', year))



In [8]:
years = ee.List.sequence(2016, 2025)

s2 = ee.ImageCollection.fromImages(years.map(get_yearly_s2))
s1 = ee.ImageCollection.fromImages(years.map(get_yearly_s1))
nl = ee.ImageCollection.fromImages(years.map(get_yearly_viirs))

s2

In [9]:
# ============================================
gdp_hdi = ee.Image("projects/sat-io/open-datasets/GRIDDED_HDI_GDP/total_gdp_perCapita_1990_2020_30arcsec").clip(aoi)
gdp_hdi

In [10]:
Map = geemap.Map()
Map.centerObject(aoi, 10)
Map.addLayer(aoi, {'color': 'red'}, 'AOI')
Map.addLayer(gdp_hdi.select('PPP_2020'), {'min': 0, 'max': 5000000, 'palette': ['blue', 'yellow', 'red']}, 'GDP PPP')

    
# Map.addLayer(s2.first(), 
#             {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, "Sentinel-2 RGB", False)

Map.addLayer(nl.first(), 
            {'min': 0, 'max': 50, 'palette': ['black', 'yellow', 'white']}, "Nightlight", False)

Map


Map(center=[14.754626398561136, -17.26525114764092], controls=(WidgetControl(options=['position', 'transparent…

In [11]:
ee.batch.Export.image.toDrive(
        image=gdp_hdi,
        description=f'{cityname}_grid_gdp',
        folder='GEE_Exports',
        fileNamePrefix=f'{cityname}_grid_gdp',
        region=aoi.geometry(),
        scale=10,
        maxPixels=1e13
    ).start()

In [12]:
# Export each year as separate TIF

for year in range(2016, 2025):
    img_s1 = s1.filter(ee.Filter.eq('year', year)).first()
    img_s2 = s2.filter(ee.Filter.eq('year', year)).first()
    img_nl = nl.filter(ee.Filter.eq('year', year)).first()

    ee.batch.Export.image.toDrive(
        image=img_s2,
        description=f'{cityname}_s2_{year}',
        folder='GEE_Exports',
        fileNamePrefix=f's2_{year}',
        region=aoi.geometry(),
        scale=10,
        maxPixels=1e13
    ).start()
    
    ee.batch.Export.image.toDrive(
        image=img_s1,
        description=f'{cityname}_s1_{year}',
        folder='GEE_Exports',
        fileNamePrefix=f'{cityname}_s1_{year}',
        region=aoi.geometry(),
        scale=10,
        maxPixels=1e13
    ).start()
    
    ee.batch.Export.image.toDrive(
        image=img_nl,
        description=f'{cityname}_nightlight_{year}',
        folder='GEE_Exports',
        fileNamePrefix=f'{cityname}_nightlight_{year}',
        region=aoi.geometry(),
        scale=500,
        maxPixels=1e13
    ).start()

print('All exports started')

All exports started
